# Data ingestion pipeline

## [Contents](#contents)

- [1. Overview](#1-overview)

- [2. Environment Configuration](#2-environment-configuration)

- [3. Crop Data](#3-crop-data)
  - [3.1 Download the .ods object from the resource URL](#31-download-the-ods-object-from-the-resource-url)
  - [3.2 Create list of all sheets in the .ods workbook](#32-create-list-of-all-sheets-in-the-ods-workbook)
  - [3.3 3.3 Parse each sheet into separate raw data .csv](#33-parse-each-sheet-into-separate-raw-data-csv)
  - [3.4 Further cleanse raw .csv for loading into Pandas dataframe](#34-further-cleanse-raw-csv-for-loading-into-pandas-dataframe)
  - [3.5 Parse cleansed .csv into Pandas for further wrangling and SQLite loading](#35-parse-cleansed-csv-into-pandas-for-further-wrangling-and-sqlite-loading)

- [4. Weather Data](#4-weather-data)
  - [4.1 Resource URL](#41-resource-url)
  - [4.2 Weather types and Areas](#42-weather-types-and-areas)
  - [4.3 Parse weather data into local .csv](#43-parse-weather-data-into-local-csv)
  - [4.4 Load into SQLite](#44-load-into-sqlite)

## [1. Overview](#contents)

This notebook is concerned with the extraction, transformation and loading of publicly available data into a database to provide datasets for downstream analytical query.

Two datasets are ingested:

- Weather data from the UK Meteorological service.
- Crop yield data from the UK Department for Environment, Farming and Rural Affairs (DEFRA).

Each dataset will be first downloaded to local directories in the form of .csv. This will allow for continued processing without having need of an internet connection.

Once raw data has been downloaded into raw .csv files, these will be parsed into Pandas Dataframes for wrangling into formats suitable for writing to a simple SQLite database.

The resulting database will then be available for the next analytical jupyter notebook.

## [2. Environment Configuration](#contents)

This needs to be run at the beginning of any new session to ensure the correct libraries are installed.

In [ ]:
# For calling data repository URL
import requests
import io

# for file and directory manipulation
import os

# For manipulating data with Pandas
import pandas as pd

# using the sqlite3 database
import sqlite3

# Ensure the odfpy library is installed
%pip install odfpy

## [3. Crop Data](#contents)

### 3.1 Download the .ods object from the resource URL

In [ ]:
# N.B. Need to ensure odfpy is installed in the environment. This allows the ODS binary to be parsed.

# Creates a ByteIO Class Object

url = "https://assets.publishing.service.gov.uk/media/68e778c1e5f463a62cb98588/UK-cops-webseries-251009a.ods"
try:
    response = requests.get(url, timeout=30)
    response.raise_for_status()  # Raises for 4xx/5xx codes.
    buffer_object = io.BytesIO(response.content)
    print(f"Download succeeded: {len(response.content)} bytes")
except requests.exceptions.RequestException as e:
    buffer_object = None
    print(f"ERROR: Unable to download ODS file from {url}: {e}")
except Exception as e:
    buffer_object = None
    print(f"Unexpected error while creating buffer_object: {e}")

### 3.2 Create list of all sheets in the .ods workbook

In [ ]:
# https://pandas.pydata.org/docs/dev/reference/api/pandas.ExcelFile.sheet_names.html
# this outputs an ExcelFile Class Object

list_odf_sheets = []

if buffer_object is None:
    raise RuntimeError("buffer_object is not available; cannot read workbook")

try:
    crops_file = pd.ExcelFile(path_or_buffer=buffer_object, engine="odf")
    list_odf_sheets = list(crops_file.sheet_names)
    print(f"Sheets found: {len(list_odf_sheets)}")
except Exception as e:
    list_odf_sheets = []
    print(f"ERROR: Unable to read sheets from ODS workbook: {e}")

list_odf_sheets


### 3.3 Parse each sheet into separate raw data .csv

For each sheet with suffix 'Region' parse into from **buffer_object** into dataframes and output as local .csv files.

Objects:
- label: **raw_wheat.csv**, object type: **.csv**
- label: **raw_winter_barley.csv**, object type: **.csv**
- label: **raw_spring_barley.csv**, object type: **.csv**
- label: **raw_total_barley.csv**, object type: **.csv**
- label: **raw_oats.csv**, object type: **.csv**
- label: **raw_osr.csv**, object type: **.csv**

Locally saving the file allows decoupling the need for network calls, allowing for faster I/O operations. This also allows me to continue development whilst internet connection was not availalble, e.g. whilst travelling on the train.

Exporting locally to .csv files does come at the cost of storage. This can be mitigated by deleting the files once processing is completed, retaining only the required data for reference. However, the size of data is very small and easily maintained on even the most limited of systems.

In [ ]:
# Define the prefix
prefix = 'Regional'

# Ensure that the directory exists before writing files
os.makedirs('data/crops/raw', exist_ok=True)

if not list_odf_sheets:
    print("WARNING: No sheets detected. Skipping raw CSV export step.")

# Use for loop to list through list of sheet names, parse sheet into dataframe, and output locally as .csv file.
for name in list_odf_sheets:
    if name.startswith(prefix):
        try:
            df_sheet = pd.ExcelFile(path_or_buffer=buffer_object, engine="odf").parse(name)
            output_path = f'data/crops/raw/{name}.csv'
            df_sheet.to_csv(output_path, index=False)
            print(f"Exported {name} -> {output_path}")
        except Exception as e:
            print(f"ERROR: failed to export sheet {name}: {e}")


### 3.4 Further cleanse raw .csv for loading into Pandas dataframe

- For each sheet CSV:
  - load into dataframe.
  - crop to new dataframe with relevant rows and columns.
  - Reset column headers and index to years.
- Save as csv in "yeilds" directory.

#### 3.4.1 List of target .csv files

In [ ]:
crop_file_names = [
			'Regional_oats',
			'Regional_OSR',
			'Regional_spring_barley',
			'Regional_total_barley',
			'Regional_wheat',
			'Regional_winter_barley'
]

#### 3.4.2 Loop through each sheet and output formatted .csv

This is a consolidated process for apply this across all the crop files at once.

In [ ]:
# parameterise start and end rows for yield section.
yield_headers_start = 17
yield_rows = 13

for crop in crop_file_names:
    try:
        file_path = f'data/crops/raw/{crop}.csv'
        df_raw = pd.read_csv(file_path, header=yield_headers_start, nrows=yield_rows)

        # Drop unwanted columns.
        unwanted_columns = ["% change 2025/2024", "2025 confidence interval", "Unnamed: 30", "5 year average"]
        df_crop = df_raw.drop(columns=[col for col in unwanted_columns if col in df_raw.columns], errors='ignore')

        # Rename columns for consistency.
        df_crop.rename(columns={
            'Yields (1)': 'Year',
            '2022(1)': '2022',
            '2022(a)': '2022',
        }, inplace=True)

        # Transpose dataframe with regions as columns and years as rows.
        df_long = df_crop.transpose()

        # Make the first row, row index 0, the column names.
        if df_long.empty:
            raise ValueError(f"Extracted table empty for crop {crop}")

        df_long.columns = df_long.iloc[0]

        # Reset the dataframe to start without the first row that has now become the column names.
        df_long = df_long[1:].copy()

        # Clear the column heading name.
        df_long.columns.name = None

        # Reset the index to make years a regular column
        df_long.reset_index(inplace=True)

        # Rename new column for clarity (the index becomes a column named 'index' by default)
        df_long.rename(columns={'index': 'Year'}, inplace=True)

        # Convert data types
        for column in df_long.columns:
            if column == 'Year':
                df_long[column] = pd.to_numeric(df_long[column], errors='coerce').astype('Int64')
            else:
                df_long[column] = pd.to_numeric(df_long[column], errors='coerce').astype(float)

        # Save to "yields" directory as transformed csv file.
        os.makedirs('data/crops/yields/', exist_ok=True)

        name = f'yields_{crop[9:]}'
        df_long.to_csv(f'data/crops/yields/{name}.csv', index=False)
        print(f"Processed and saved {name}.csv")

    except FileNotFoundError:
        print(f"ERROR: raw file not found for crop {crop}: {file_path}")
    except Exception as e:
        print(f"ERROR processing crop {crop}: {e}")


### 3.5 Parse cleansed .csv into Pandas for further wrangling and SQLite loading

#### 3.5.1 Prerequisites for SQLite

In [ ]:
os.makedirs('data', exist_ok=True)
project_db  = 'data/project_db.db'

#### 3.5.2 Load all formatted .csv into SQLite database

In [ ]:
# Get all file names to loop through.
yield_file_list: list = os.listdir('data/crops/yields')

print(yield_file_list)

In [ ]:
for yield_file in yield_file_list:

	# Parse file to dataframe.
	df_yield = pd.read_csv(f'data/crops/yields/{yield_file}')

	# Truncate file name for table name.
	table_name = yield_file[:-4]

	# open connection to database.
	conn = sqlite3.connect(project_db)

	try:
		# Load dataframe to sqlite.
		df_yield.to_sql(
			name=table_name,
			con=conn,
			if_exists='replace',
			index=False
		)

		print(f'Table {table_name} updated')

	except Exception as e:
		print(f"Error: {(str(e))}")
	
	finally:
		# Close connection
		conn.close()


#### 3.5.3 Test database query

In [ ]:
conn = sqlite3.connect(project_db)

df_db_sp_yields = pd.read_sql('SELECT * FROM yields_spring_barley', conn)
conn.close()

df_db_sp_yields.head()

## [4. Weather Data](#contents)

### 4.1 Resource URL

For the purposes of retrieving weather data by year, to select the data to download, two parameters are passed: the **Weather Type**, and the **Area**. This information was manually extracted from the html of the website: <https://www.metoffice.gov.uk/research/climate/maps-and-data/uk-and-regional-series>

The for requests follows the format <https://www.metoffice.gov.uk/pub/data/weather/uk/climate/datasets/{weather}/date/{region}.txt>. Note the **weather** and **area** parameters in curly braces. These are used to determine which specific dataset is requested.

### 4.2 Weather types and Areas

Data retrieved from the url specifies a weather type and region. These can be captured from inspecting the website.

#### 4.2.1 Weather data

These are the parameters used.

|Parameter|Label|
|---------|-----|
|Max temp|Tmax|
|Min temp|Tmin|
|Mean temp|Tmean|
|Sunshine|Sunshine|
|Rainfall|Rainfall|
|Rain days ≥1.0mm|Raindays1mm|
|Days of air frost|AirFrost|

This "weather_types" list  can be used to iterate over to retrieve all weather types.

In [ ]:
weather_types = ["Tmax", "Tmin", "Tmean", "Sunshine", "Rainfall", "Raindays1mm", "AirFrost"]

print(weather_types)

#### 4.2.2 Regions

These Regions are how the data for each weather type is aggregated and segmented.

|ID|Region|Label|
|--|------|-----|
|1|UK|UK|
|2|England|England|
|3|Wales|Wales|
|4|Scotland|Scotland|
|5|Northern Ireland|Northern_Ireland|
|6|England & Wales|England_and_Wales|
|7|England N|England_N|
|8|Englan S|Englan_S|
|9|Scotland N|Scotland_N|
|10|Scotland E|Scotland_E|
|11|Scotland W|Scotland_W|
|12|England E & NE|England_E_and_NE|
|13|England NW/Wales N|England_NW_and_N_Wales|
|14|Midlands|Midlands|
|15|East Anglia|East_Anglia|
|16|England SW/Wales S|England_SW_and_S_Wales|
|17|England SE/Central S|England_SE_and_Central_S|

This "areas" list can be used to iterate to request each of the areas in turn for each of the weather types previously noted.

In [ ]:
areas = ["UK",
		"England",
		"Wales",
		"Scotland",
		"Northern_Ireland",
		"England_and_Wales",
		"England_N",
		"England_S",
		"Scotland_N",
		"Scotland_E",
		"Scotland_W",
		"England_E_and_NE",
		"England_NW_and_N_Wales",
		"Midlands",
		"East_Anglia",
		"England_SW_and_S_Wales",
		"England_SE_and_Central_S"]

print (areas)

### 4.3 Parse weather data into local .csv

Rather than downloading each of file for each region for each parameter, it would be better to directly retrieve the data from the API or URL.

The URL clearly indicates in plain text the Region and Parameter labels. I can extract these and inject them into the URL string used to request the data.

In [ ]:
# For each Weather Type, for each Area, request data from the met office website.
for weather in weather_types:
	for area in areas:

		# Ensure that the directory exists before writing files
		os.makedirs(f'data/weather/{weather}', exist_ok=True)

		# Get data with parametrised URL.
		url= f"https://www.metoffice.gov.uk/pub/data/weather/uk/climate/datasets/{weather}/date/{area}.txt"
		response = requests.get(url)
		print (response.text)

		# Read with flexible whitespace handling.
		try:
			df = pd.read_csv(url, 
							 sep='\s+',					# Handle variable spacing
							 skipinitialspace=True,		# Skip leading spaces
							 skip_blank_lines=True,		# Skip empty lines
							 skiprows=5)				# Skip the first 5 lines of metadata

			# Write to CSV with a filename that includes the weather type and area.
			df.to_csv(f"data/weather/{weather}/{areas}_{weather}.csv", index=False)

		# Catch any exceptions.
		except Exception as e:
			print(f"Error: {str(e)}")

### 4.4 Load into SQLite

#### 4.4.1 Prerequisites for SQLite

In [ ]:
os.makedirs('data', exist_ok=True)
project_db  = 'data/project_db.db'

#### 4.4.2 Load .csv into SQLite database

For offline loading, the downloaded .csv can be loaded into the SQLite database.

In [ ]:
# function to loop through both lists and retrieve data from url.
def loop_csv_local():

	conn = sqlite3.connect(project_db)
	
	for weather in weather_types:
		for i, area in enumerate(areas):

			# path to the previously downloaded .csv
			path = f"data/weather/{weather}/{area}_{weather}.csv"
	
			try:
				df = pd.read_csv(path,
								sep=',',					# Handle variable spacing
								skipinitialspace=True,		# Skip leading spaces
								skip_blank_lines=True,		# Skip empty lines
								skiprows=0,					# Skip the first 5 lines of metadata
								na_values=['---'],			# Specifies which value indicates null/missing data.
								keep_default_na=True,		# Retain default NA values.
								dtype={'year':int}
				)
	
				# Add an 'area' column to the DataFrame so this can be queried independently.
				df['area'] = area
	
				# Replace the table on the first area (clears old data),
				# then append for the rest — avoids duplicates on re-runs.
				write_mode = 'replace' if i == 0 else 'append'
	
				df.to_sql(
					name=weather,
					con=conn,
					if_exists=write_mode,
					index=False
				)
	
				# Write to console to monitor progress.
				print(f"Table {weather} / {area} updated.")
	
			# Write to console to show if any errors occur, but continue processing the rest of the data.
			except Exception as e:
				print(f"Error with updating table ({weather}/{area}): {str(e)}.")
	
	conn.close()
	
	print("All tables updated successfully.")

response = input("Do you wish to proceed with loading SQLite directly from locally saved .csv files?: y /n")

if response.lower() == 'y':
	print('Proceeding with download...')
	loop_csv_local()
	print('Database updated.')

elif response.lower() =='n':
	print('Action aborted.')

else:
	print('Incorrect input. Try again.')

#### 4.4.3 Alternatively load url direct into SQLite database

The URL clearly indicates in plain text the Region and Parameter labels. I can extract these and inject them into the URL string used to request the data.

In [ ]:
# function to loop through both lists and retrieve data from url.
def loop_url_remote():

	conn = sqlite3.connect(project_db)
	
	for weather in weather_types:
		for i, area in enumerate(areas):
		
			# Get data with parametrised URL.
			url = f"https://www.metoffice.gov.uk/pub/data/weather/uk/climate/datasets/{weather}/date/{area}.txt"
	
			# Read with flexible whitespace handling.
			try:
				df = pd.read_csv(url,
								sep='\s+',					# Handle variable spacing
								skipinitialspace=True,		# Skip leading spaces
								skip_blank_lines=True,		# Skip empty lines
								skiprows=5,					# Skip the first 5 lines of metadata
								na_values=['---'],			# Specifies which value indicates null/missing data.
								keep_default_na=True,		# Retain default NA values.
								dtype={'year':int}
				)
	
				# Add an 'area' column to the DataFrame so this can be queried independently.
				df['area'] = area
	
				# Replace the table on the first area (clears old data),
				# then append for the rest — avoids duplicates on re-runs.
				write_mode = 'replace' if i == 0 else 'append'
	
				df.to_sql(
					name=weather,
					con=conn,
					if_exists=write_mode,
					index=False
				)
	
				# Write to console to monitor progress.
				print(f"Table {weather} / {area} updated.")
	
			# Write to console to show if any errors occur, but continue processing the rest of the data.
			except Exception as e:
				print(f"Error with updating table ({weather}/{area}): {str(e)}.")
	
	conn.close()
	
	print("All tables updated successfully.")

response = input("Do you wish to proceed with loading SQLite directly from the url resource?: y /n")

if response.lower() == 'y':

	print('Proceeding with download...')
	loop_url_remote()
	print('Database updated.')

elif response.lower() =='n':

	print('Action aborted.')

else:
	print('Incorrect input. Try again.')

#### 4.4.4 Test SQLite query

In [ ]:
conn = sqlite3.connect(project_db)

df_rainfall_all = pd.read_sql('SELECT * FROM Rainfall ', conn)
conn.close()

df_rainfall_all